# Support Integrity Auditor — Severity Mismatch Classifier (Prototype 2)

**Model:** fine-tuned `microsoft/deberta-v3-small` (a full fine-tune of a small encoder).

This satisfies the requirement of using a *fine-tuned / adapter-trained* model rather than a
frozen zero-shot pipeline: every backbone weight is updated on our labelled data.

Task: binary classification — does a ticket's stated `Priority_Level` mismatch the
severity our pseudo-labelling pipeline inferred (`Is_Mismatch` = 0/1).

In [9]:
import os
# Set before torch/CUDA init: reduces VRAM fragmentation on the 4GB GPU.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

CUDA available: True
Device: NVIDIA GeForce RTX 3050 Laptop GPU


## 1. Load data & build the input text

In [27]:
df = pd.read_csv("../dataset/preprocessed_data.csv")
print(f"Loaded {len(df)} rows")

def to_text(row):
    return (
        f"{row['Ticket_Subject']}: {row['Ticket_Description']} "
    )

df["TXT"] = df.apply(to_text, axis=1)
dataset = df[["TXT", "Is_Mismatch"]].rename(columns={"Is_Mismatch": "labels"})

print("\nLabel balance:")
print(dataset["labels"].value_counts(normalize=True).round(3))
dataset.head()

Loaded 20000 rows

Label balance:
labels
1    0.654
0    0.346
Name: proportion, dtype: float64


,TXT,labels
0,"Hours of operation - Individual: Hi Support, W...",1
1,"Data not syncing - Card: Hi Support, The appli...",1
2,"2FA issues - Question: Hi Support, How do I up...",0
3,"Login failed - Let: Hi Support, The dashboard ...",1
4,"Refund status - Attention: Hi Support, I have ...",1


## 2. Train / eval split

In [28]:
train_df, eval_df = train_test_split(
    dataset, test_size=0.2, random_state=42, stratify=dataset["labels"]
)
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
eval_dataset  = Dataset.from_pandas(eval_df,  preserve_index=False)
print(len(train_dataset), "train /", len(eval_dataset), "eval")

16000 train / 4000 eval


## 3. Tokenizer & model — `deberta-v3-small`

In [29]:
model_name = "microsoft/deberta-v3-small"

# DeBERTa-v3 uses a SentencePiece tokenizer (needs the `sentencepiece` package, already installed).
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
)
# Force fp32 weights BEFORE training. The checkpoint ships in fp16, and fp16=True mixed
# precision on fp16 master weights raises "Attempting to unscale FP16 gradients".
# Mixed precision needs fp32 master weights + an autocast fp16 forward pass.
model = model.float().to("cuda")

n_total = sum(p.numel() for p in model.parameters())
print(f"{model_name}: {n_total/1e6:.1f}M params (all trainable - full fine-tune)")

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 31941.09it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING  

microsoft/deberta-v3-small: 141.9M params (all trainable - full fine-tune)


In [30]:
def tokenize_function(batch):
    return tokenizer(batch["TXT"], truncation=True, max_length=128)  # tickets are ~55 tokens

tokenized_train = train_dataset.map(tokenize_function, batched=True).remove_columns(["TXT"])
tokenized_eval  = eval_dataset.map(tokenize_function,  batched=True).remove_columns(["TXT"])

tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_eval.set_format("torch",  columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)  # dynamic padding per batch

Map: 100%|██████████| 4000/4000 [00:00<00:00, 33681.95 examples/s]


## 4. Metrics

In [31]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    # also the positive-class (mismatch=1) F1 — the metric we actually care about
    _, _, f1_pos, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1, zero_division=0)
    return {"accuracy": acc, "f1": f1, "f1_mismatch": f1_pos, "precision": p, "recall": r}

## 5. Training arguments & Trainer

In [32]:
training_args = TrainingArguments(
    output_dir="./deberta_mismatch",
    num_train_epochs=3,
    per_device_train_batch_size=8,       # 4GB GPU: 8 + accumulation keeps peak ~2.3GB
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,       # effective batch = 16
    learning_rate=2e-5,                  # standard full fine-tune LR for encoders
    warmup_steps=100,
    weight_decay=0.01,
    optim="adamw_bnb_8bit",              # 8-bit Adam: ~halves optimizer-state VRAM vs fp32 Adam
    logging_dir="./logs",
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_mismatch", # optimise positive-class F1, not just weighted
    greater_is_better=True,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## 6. Train

In [33]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,F1 Mismatch,Precision,Recall
1,1.140287,0.600393,0.665750,0.659880,0.752728,0.656576,0.665750
2,1.176291,0.584994,0.665750,0.665081,0.745576,0.664456,0.665750
3,1.151704,0.582972,0.659750,0.629199,0.767151,0.632254,0.659750


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s]


TrainOutput(global_step=3000, training_loss=1.1840194555918375, metrics={'train_runtime': 401.4273, 'train_samples_per_second': 119.573, 'train_steps_per_second': 7.473, 'total_flos': 385700848874880.0, 'train_loss': 1.1840194555918375, 'epoch': 3.0})

## 7. Evaluate — full classification report & confusion matrix

In [17]:
pred = trainer.predict(tokenized_eval)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=1)

print("Eval metrics:", {k: round(v, 4) for k, v in pred.metrics.items() if k.startswith("test_")})
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["match (0)", "mismatch (1)"], digits=4))
print("Confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(y_true, y_pred))

Eval metrics: {'test_loss': 0.5792, 'test_accuracy': 0.6647, 'test_f1': 0.6557, 'test_f1_mismatch': 0.7554, 'test_precision': 0.652, 'test_recall': 0.6647, 'test_runtime': 4.6092, 'test_samples_per_second': 867.823, 'test_steps_per_second': 27.119}

Classification report:
              precision    recall  f1-score   support

   match (0)     0.5190    0.4249    0.4672      1384
mismatch (1)     0.7224    0.7917    0.7554      2616

    accuracy                         0.6647      4000
   macro avg     0.6207    0.6083    0.6113      4000
weighted avg     0.6520    0.6647    0.6557      4000

Confusion matrix [rows=true, cols=pred]:
[[ 588  796]
 [ 545 2071]]


> **Caveat:** `Is_Mismatch` is a *pseudo-label* derived from our own scoring heuristic.
> A high F1 means the model has learned to reproduce that heuristic — it is not a ground-truth
> accuracy measure. Treat it as "did the adapter learn the signal," not "is the rule correct."

## 8. Save the fine-tuned model

In [18]:
save_dir = "./deberta_mismatch/best"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved to", save_dir)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s]

Saved to ./deberta_mismatch/best
